# <font color="#418FDE" size="6.5" uppercase>**Regression vergleichen**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Trainieren lineare, regularisierte und robuste Regressionsmodelle in Pipelines. 
- Vergleichen nichtlineare Regressoren wie k-NN, Bäume, Random Forest, Boosting und SVR. 
- Bewerten Regressionsmodelle mit MAE, RMSE, R², Lernkurven und Laufzeit. 


## **1. Lineare Regressionen**

### **1.1. Mehrere Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_01_01.jpg?v=1787643893" width="250">



>* Vorhersagen nutzen mehrere Einflussgrößen gemeinsam
>* Einfache Modelle bleiben schnell und interpretierbar

>* Pipelines bereiten gemischte Merkmale zuverlässig vor
>* Gleiche Schritte verhindern Datenleckage und verzerrte Vergleiche

>* Koeffizienten im Kontext anderer Merkmale deuten
>* Datenstruktur auf Redundanz kritisch prüfen



In [ ]:
#@title Python-Code - Mehrere Merkmale

# Wir trainieren eine multiple lineare Regression.
# Eine Pipeline verarbeitet Merkmale sauber gemeinsam.
# Die Ausgabe zeigt Fehler und Koeffizienten.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine Daten mit mehreren Merkmalen.
features, target, true_weights = make_regression(
    n_samples=160,
    n_features=3,
    noise=12.0,
    coef=True,
    random_state=42,
)

# Die Merkmalsnamen machen die Koeffizienten lesbarer.
feature_names = np.array(["Wohnfläche", "Zimmer", "Baujahr"])
if features.shape != (160, 3):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Fehlende Werte simulieren einen typischen Praxisschritt.
features = features.copy()
features[0, 1] = np.nan
features[7, 2] = np.nan

# Der Split trennt Training und Test fair.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Die Pipeline imputiert, skaliert und trainiert nur mit Trainingsdaten.
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer()), ("scaler", StandardScaler())]
)
preprocessor = ColumnTransformer(
    transformers=[("numeric", numeric_pipeline, [0, 1, 2])]
)

# Das lineare Modell nutzt alle drei Merkmale gleichzeitig.
model = Pipeline(
    steps=[("preprocessor", preprocessor), ("regressor", LinearRegression())]
)
model.fit(X_train, y_train)

# Wir bewerten Vorhersagen auf ungesehenen Testdaten.
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = model.score(X_test, y_test)
coefficients = model.named_steps["regressor"].coef_

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Test-MAE: {mae:.1f}")
print(f"Test-R²: {r2:.2f}")
for name, value in zip(feature_names, coefficients):
    print(f"Koeffizient für {name}: {value:.1f}")

# Das Streudiagramm vergleicht echte und vorhergesagte Zielwerte.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(y_test, predictions, alpha=0.75, label="Testpunkte")
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], label="Ideal")
ax.set_title("Multiple lineare Regression in einer Pipeline")
ax.set_xlabel("Echter Zielwert")
ax.set_ylabel("Vorhergesagter Zielwert")
ax.legend()
plt.show()



### **1.2. Regularisierte Modelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_01_02.jpg?v=1787643896" width="250">



>* Regularisierung verhindert Überanpassung durch kleinere Koeffizienten
>* Pipelines sichern korrekte Skalierung und Training

>* Ridge, Lasso und Elastic Net vergleichen.
>* Modellwahl per Kreuzvalidierung systematisch prüfen.

>* Koeffizienten vorsichtig und kontextabhängig interpretieren
>* Modelle nach Validierung und Plausibilität vergleichen



In [ ]:
#@title Python-Code - Regularisierte Modelle

# Wir vergleichen Ridge und Lasso in Pipelines.
# Regularisierung braucht skalierte Merkmale für faire Koeffizienten.
# Die Ausgabe zeigt Testfehler und Koeffizienten.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes

from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen Regressionsdatensatz aus scikit-learn.
data = load_diabetes()
X = data.data
y = data.target

# Diese Prüfung macht die Beispielannahmen sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der Testteil bleibt bis zur Bewertung unberührt.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Beide Modelle nutzen dieselbe Skalierung in einer Pipeline.
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
lasso_model = make_pipeline(StandardScaler(), Lasso(alpha=1.0, max_iter=5000))

# Jetzt trainieren wir die regularisierten Modelle.
ridge_model.fit(X_train, y_train)
lasso_model.fit(X_train, y_train)

# Vorhersagen auf Testdaten zeigen die Generalisierung.
ridge_predictions = ridge_model.predict(X_test)
lasso_predictions = lasso_model.predict(X_test)

# MAE ist ein leicht verständlicher durchschnittlicher Fehler.
ridge_mae = mean_absolute_error(y_test, ridge_predictions)
lasso_mae = mean_absolute_error(y_test, lasso_predictions)

# Lasso kann Koeffizienten exakt auf null setzen.
ridge_coefficients = ridge_model.named_steps["ridge"].coef_
lasso_coefficients = lasso_model.named_steps["lasso"].coef_

# Wir zählen, wie viele Merkmale aktiv bleiben.
ridge_active = np.count_nonzero(np.abs(ridge_coefficients) > 0.001)
lasso_active = np.count_nonzero(np.abs(lasso_coefficients) > 0.001)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Ridge MAE auf Testdaten: {ridge_mae:.1f}")
print(f"Lasso MAE auf Testdaten: {lasso_mae:.1f}")
print(f"Aktive Merkmale: Ridge {ridge_active}, Lasso {lasso_active}")

# Das Diagramm vergleicht die gelernten Koeffizienten.
feature_numbers = np.arange(1, X.shape[1] + 1)
plt.figure(figsize=(8, 4))
plt.plot(feature_numbers, ridge_coefficients, marker="o", label="Ridge")

plt.plot(feature_numbers, lasso_coefficients, marker="s", label="Lasso")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Regularisierte Koeffizienten im Vergleich")
plt.xlabel("Merkmalnummer")

plt.ylabel("Koeffizient nach Skalierung")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Residuen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_01_03.jpg?v=1787643895" width="250">



>* Residuen zeigen Abweichungen zwischen Beobachtung und Vorhersage
>* Muster weisen auf Modellgrenzen oder fehlende Merkmale

>* Residuen sollten zufällig und musterlos streuen
>* Muster zeigen Modellgrenzen und Verbesserungsbedarf

>* Große Residuen zeigen mögliche Ausreißer.
>* Pipelines und Kontext sorgfältig prüfen.



In [ ]:
#@title Python-Code - Residuen verstehen

# Dieses Beispiel macht Residuen sichtbar.
# Eine Pipeline trainiert lineare Regression.
# Das Diagramm zeigt Fehlerstrukturen deutlich.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine, reproduzierbare Regressionsdaten.
features, target = make_regression(
    n_samples=120,
    n_features=1,
    noise=18,
    random_state=42,
)

# Eine leichte Krümmung erzeugt ein erkennbares Residuenmuster.
target = target + 0.08 * features[:, 0] ** 2

# Diese Prüfung schützt vor unerwarteten Datenformen.
if features.shape[0] != target.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der Split trennt Training und faire Prüfung.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    random_state=42,
)

# Die Pipeline skaliert nur mit Trainingsdaten.
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

# Residuen sind beobachtete Werte minus Vorhersagen.
predictions = model.predict(X_test)
residuals = y_test - predictions
mae = mean_absolute_error(y_test, predictions)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Mittlerer absoluter Fehler: {mae:.2f}")
print(f"Durchschnittliches Residuum: {np.mean(residuals):.2f}")
print("Muster im Plot deuten auf fehlende Nichtlinearität hin.")

# Ein Residuenplot zeigt, ob Fehler zufällig streuen.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(predictions, residuals, alpha=0.75, label="Testpunkte")
ax.axhline(0, color="black", linewidth=1, label="Null-Residuum")

ax.set_title("Residuen einer linearen Regression")
ax.set_xlabel("Vorhergesagter Zielwert")
ax.set_ylabel("Residuum: beobachtet minus vorhergesagt")
ax.legend()
plt.show()



## **2. Nichtlineare Regression**

### **2.1. Polynome und Splines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_02_01.jpg?v=1787643889" width="250">



>* Gekrümmte Merkmale erfassen nichtlineare Zusammenhänge
>* Hohe Polynomgrade erhöhen Overfitting-Gefahr

>* Splines teilen Merkmalsbereiche in glatte Abschnitte
>* Sie modellieren lokale Muster ohne Randextreme

>* Polynome und Splines verbinden Modellwelten.
>* Validierung zeigt passende nichtlineare Verfahren.



In [ ]:
#@title Python-Code - Polynome und Splines

# Dieses Beispiel vergleicht Polynome und Splines.
# Beide Modelle lernen gekrümmte Regressionskurven.
# Testfehler zeigen Überanpassung und glatte Flexibilität.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import SplineTransformer

# Wir erzeugen kleine, gekrümmte Regressionsdaten.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 80).reshape(-1, 1)
noise = rng.normal(0, 0.35, size=x.shape[0])

# Die Zielwerte enthalten ein glattes nichtlineare Muster.
y = np.sin(x[:, 0]) + 0.25 * x[:, 0] + noise

# Eine einfache Prüfung verhindert unklare Formfehler.
if x.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte müssen gleich viele Zeilen haben.")

# Der Testteil simuliert neue, unbekannte Daten.
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

# Ein hohes Polynom kann sehr flexibel werden.
poly_model = make_pipeline(
    PolynomialFeatures(degree=9, include_bias=False), Ridge(alpha=1.0)
)

# Splines modellieren lokale glatte Abschnitte.
spline_model = make_pipeline(
    SplineTransformer(n_knots=6, degree=3), Ridge(alpha=1.0)
)

# Beide Pipelines lernen nur aus den Trainingsdaten.
poly_model.fit(x_train, y_train)
spline_model.fit(x_train, y_train)

# Wir bewerten beide Modelle auf unbekannten Testdaten.
poly_rmse = np.sqrt(mean_squared_error(y_test, poly_model.predict(x_test)))
spline_rmse = np.sqrt(mean_squared_error(y_test, spline_model.predict(x_test)))

# Ein feines Raster macht die gelernten Kurven sichtbar.
x_grid = np.linspace(0, 10, 300).reshape(-1, 1)
poly_curve = poly_model.predict(x_grid)
spline_curve = spline_model.predict(x_grid)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Test-RMSE Polynom Grad 9: {poly_rmse:.3f}")
print(f"Test-RMSE Spline: {spline_rmse:.3f}")

# Die Grafik zeigt Anpassung und mögliche Randprobleme.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_train[:, 0], y_train, s=25, alpha=0.7, label="Training")
ax.scatter(x_test[:, 0], y_test, s=25, alpha=0.7, label="Test")

ax.plot(x_grid[:, 0], poly_curve, color="tab:red", label="Polynom Grad 9")
ax.plot(x_grid[:, 0], spline_curve, color="tab:green", label="Spline")
ax.set_title("Polynomiale Regression und Splines im Vergleich")
ax.set_xlabel("Eingabemerkmal x")

ax.set_ylabel("Zielwert y")
ax.legend()
plt.show()



### **2.2. Robuste Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_02_02.jpg?v=1787643887" width="250">



>* Verlässliche Vorhersagen trotz fehlerhafter Daten
>* Ausreißereinfluss begrenzen, typische Muster bewahren

>* Ausreißer können Mietpreisprognosen stark verzerren
>* Robuste Modelle begrenzen ihren Einfluss

>* Ausreißer beeinflussen nichtlineare Modelle unterschiedlich.
>* Prüfe Fehler, Sonderfälle und echtes Signal.



In [ ]:
#@title Python-Code - Robuste Regression

# Wir vergleichen robuste und empfindliche Regression.
# Ausreißer sollen den Unterschied sichtbar machen.
# Die Grafik zeigt stabilere robuste Vorhersagen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Ein fester Zufallsgenerator macht das Beispiel reproduzierbar.
rng = np.random.default_rng(42)

# Wir erzeugen einen einfachen linearen Zusammenhang mit Rauschen.
x_clean = np.linspace(20, 100, 60)
noise = rng.normal(0, 8, size=x_clean.shape)
y_clean = 4.0 * x_clean + 50 + noise

# Einige extreme Sonderfälle simulieren fehlerhafte oder ungewöhnliche Daten.
x_outliers = np.array([25, 35, 90, 95])
y_outliers = np.array([420, 460, 120, 140])

# Die Trainingsdaten enthalten normale Punkte und Ausreißer.
x_all = np.concatenate([x_clean, x_outliers])
y_all = np.concatenate([y_clean, y_outliers])

# Diese Prüfung verhindert unpassende Formen für scikit-learn.
if x_all.shape[0] != y_all.shape[0]:
    raise ValueError("X und y müssen gleich viele Beobachtungen haben.")

# scikit-learn erwartet Merkmale als zweidimensionale Matrix.
X_all = x_all.reshape(-1, 1)
X_clean = x_clean.reshape(-1, 1)

# Lineare Regression gewichtet große Fehler stark.
linear_model = LinearRegression()
linear_model.fit(X_all, y_all)

# Huber Regression begrenzt den Einfluss sehr großer Fehler.
huber_model = HuberRegressor(epsilon=1.35, max_iter=200)
huber_model.fit(X_all, y_all)

# Wir bewerten beide Modelle auf den typischen, sauberen Punkten.
linear_pred_clean = linear_model.predict(X_clean)
huber_pred_clean = huber_model.predict(X_clean)

linear_mae = mean_absolute_error(y_clean, linear_pred_clean)
huber_mae = mean_absolute_error(y_clean, huber_pred_clean)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"MAE auf typischen Punkten, LinearRegression: {linear_mae:.1f}")
print(f"MAE auf typischen Punkten, HuberRegressor: {huber_mae:.1f}")

# Für die Linien nutzen wir gleichmäßig verteilte x-Werte.
x_grid = np.linspace(15, 105, 120)
X_grid = x_grid.reshape(-1, 1)

linear_line = linear_model.predict(X_grid)
huber_line = huber_model.predict(X_grid)

# Eine einzige Grafik zeigt Daten, Ausreißer und Modelllinien.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_clean, y_clean, label="typische Punkte", alpha=0.75)
ax.scatter(x_outliers, y_outliers, label="Ausreißer", marker="x", s=90)

ax.plot(x_grid, linear_line, label="LinearRegression", linewidth=2)
ax.plot(x_grid, huber_line, label="HuberRegressor", linewidth=2)
ax.set_title("Robuste Regression bei Ausreißern")
ax.set_xlabel("Wohnfläche in Quadratmetern")

ax.set_ylabel("Miete in Euro")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **2.3. Nachbarn und Bäume**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_02_03.jpg?v=1787643891" width="250">



>* k-NN schätzt Werte über ähnliche Fälle
>* Skalierung und Nachbarzahl stark beachten

>* Bäume teilen Daten in verständliche Entscheidungsbereiche
>* Begrenzte Tiefe verhindert instabile Überanpassung

>* Baum-Ensembles liefern robustere nichtlineare Vorhersagen
>* SVR und Modellwahl beachten Datenanforderungen



In [ ]:
#@title Python-Code - Nachbarn und Bäume

# Wir vergleichen Nachbarn und Bäume anschaulich.
# Beide Modelle lernen nichtlineare Regressionsmuster.
# Die Grafik zeigt unterschiedliche Vorhersageformen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Wir erzeugen kleine Daten mit gekrümmtem Zusammenhang.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 120)
y = np.sin(x) + 0.25 * rng.normal(size=x.shape)

# Scikit-learn erwartet Merkmale als zweidimensionale Tabelle.
X = x.reshape(-1, 1)
if X.shape != (120, 1) or y.shape != (120,):
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# Der Testbereich prüft Vorhersagen auf unbekannten Punkten.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# k-NN mittelt Zielwerte ähnlicher Nachbarn.
knn_model = KNeighborsRegressor(n_neighbors=7)
knn_model.fit(X_train, y_train)

# Ein Baum teilt den Merkmalsraum in einfache Bereiche.
tree_model = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

# Wir bewerten beide Modelle mit mittlerem absolutem Fehler.
knn_mae = mean_absolute_error(y_test, knn_model.predict(X_test))
tree_mae = mean_absolute_error(y_test, tree_model.predict(X_test))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"MAE k-NN: {knn_mae:.3f}")
print(f"MAE Baum: {tree_mae:.3f}")

# Ein feines Raster macht die Vorhersageformen sichtbar.
x_grid = np.linspace(0, 10, 300).reshape(-1, 1)
knn_curve = knn_model.predict(x_grid)
tree_curve = tree_model.predict(x_grid)

# Die Punkte sind Trainingsdaten, die Linien sind Modellvorhersagen.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train[:, 0], y_train, s=18, alpha=0.55, label="Trainingsdaten")
ax.plot(x_grid[:, 0], knn_curve, label="k-NN: lokal geglättet")
ax.plot(x_grid[:, 0], tree_curve, label="Baum: stufige Bereiche")

ax.set_title("Nichtlineare Regression mit Nachbarn und Baum")
ax.set_xlabel("Merkmal x")
ax.set_ylabel("Zielwert y")
ax.legend()
plt.show()



## **3. Regressionsmodelle bewerten**

### **3.1. Boosting und SVR**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_03_01.jpg?v=1787643900" width="250">



>* Boosting und SVR erfassen komplexe Muster
>* MAE, RMSE und R² gemeinsam bewerten

>* Lernkurven zeigen Über- oder Unteranpassung
>* Laufzeit gegen kleine Genauigkeitsgewinne abwägen

>* SVR braucht Skalierung, Kernel und Tuning
>* Vergleiche Metriken, Lernkurven und Laufzeit



In [ ]:
#@title Python-Code - Boosting und SVR

# Wir vergleichen Boosting und SVR fair.
# Metriken und Laufzeit zeigen praktische Unterschiede.
# Ein Diagramm macht typische Fehler sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

# Wir erzeugen kleine nichtlineare Regressionsdaten.
rng = np.random.default_rng(42)
features, target = make_regression(
    n_samples=700, n_features=6, noise=18.0, random_state=42
)

# Eine Sinuskomponente macht das Problem nichtlinear.
target = target + 45.0 * np.sin(features[:, 0])
if features.shape != (700, 6) or target.shape[0] != 700:
    raise ValueError("Die erzeugten Daten haben eine unerwartete Form.")

# Beide Modelle erhalten dieselbe faire Datenaufteilung.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Boosting lernt schrittweise Korrekturen einfacher Modelle.
boosting_model = HistGradientBoostingRegressor(
    max_iter=120, learning_rate=0.08, random_state=42
)
boosting_model.fit(X_train, y_train)
boosting_predictions = boosting_model.predict(X_test)

# SVR braucht Skalierung, deshalb nutzen wir eine Pipeline.
svr_model = make_pipeline(
    StandardScaler(), SVR(kernel="rbf", C=80.0, epsilon=8.0)
)
svr_model.fit(X_train, y_train)
svr_predictions = svr_model.predict(X_test)

# Wir berechnen drei Kennzahlen für beide Modelle.
boosting_mae = mean_absolute_error(y_test, boosting_predictions)
boosting_rmse = mean_squared_error(y_test, boosting_predictions) ** 0.5
boosting_r2 = r2_score(y_test, boosting_predictions)

# Dieselben Kennzahlen machen den SVR vergleichbar.
svr_mae = mean_absolute_error(y_test, svr_predictions)
svr_rmse = mean_squared_error(y_test, svr_predictions) ** 0.5
svr_r2 = r2_score(y_test, svr_predictions)

# Die Ausgabe bleibt kurz und lernzielbezogen.
print(f"scikit-learn Version: {sklearn.__version__}")
print("Modellvergleich auf denselben Testdaten:")
print(f"Boosting: MAE={boosting_mae:.1f}, RMSE={boosting_rmse:.1f}, R²={boosting_r2:.3f}")
print(f"SVR:      MAE={svr_mae:.1f}, RMSE={svr_rmse:.1f}, R²={svr_r2:.3f}")

# Absolute Fehler zeigen die typische Streuung der Vorhersagen.
boosting_errors = np.abs(y_test - boosting_predictions)
svr_errors = np.abs(y_test - svr_predictions)

# Ein Boxplot vergleicht die Fehlerverteilungen kompakt.
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([boosting_errors, svr_errors], tick_labels=["Boosting", "SVR"])
ax.set_title("Absolute Testfehler: Boosting gegen SVR")
ax.set_ylabel("Absoluter Fehler in Zieleinheiten")
plt.show()



### **3.2. Lernkurven interpretieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_03_02.jpg?v=1787643898" width="250">



>* Lernkurven zeigen Leistung bei wachsender Datenmenge
>* Große Leistungslücken deuten auf Überanpassung hin

>* Kurvenabstand zeigt Unteranpassung oder hohe Varianz
>* Steigende Validierungskurve spricht für mehr Daten

>* Metrik beeinflusst die Lernkurven-Deutung
>* Kontext, Laufzeit und Stabilität mitbewerten



In [ ]:
#@title Python-Code - Lernkurven interpretieren

# Wir untersuchen Lernkurven für ein Regressionsmodell.
# Trainingsgröße zeigt Unteranpassung oder Überanpassung.
# Die Grafik vergleicht Training und Validierung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.model_selection import learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

# Ein kleiner Datensatz macht die Lernkurve schnell berechenbar.
rng = np.random.default_rng(42)
X = rng.uniform(-3.0, 3.0, size=(180, 1))
y = 2.0 * X[:, 0] ** 2 + rng.normal(0.0, 2.0, size=180)

# Diese Prüfung verhindert unklare Fehler bei falschen Formen.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte müssen gleich viele Zeilen haben.")

# Ein flexibles Modell kann bei kleinen Datenmengen überanpassen.
model = make_pipeline(
    PolynomialFeatures(degree=8, include_bias=False),
    Ridge(alpha=1.0),
)

# Negative RMSE-Werte werden von scikit-learn als Score verwendet.
train_sizes, train_scores, valid_scores = learning_curve(
    model,
    X,
    y,
    train_sizes=np.linspace(0.15, 1.0, 6),
    cv=5,
    scoring="neg_root_mean_squared_error",
)

# Wir wandeln die Scores in verständliche RMSE-Fehler um.
train_rmse = -train_scores.mean(axis=1)
valid_rmse = -valid_scores.mean(axis=1)
gap = valid_rmse[-1] - train_rmse[-1]

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Letzter Trainings-RMSE: {train_rmse[-1]:.2f}")
print(f"Letzter Validierungs-RMSE: {valid_rmse[-1]:.2f}")
print(f"Abstand am Ende: {gap:.2f}")

# Die Kurven zeigen, ob mehr Daten die Validierung verbessern.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_rmse, marker="o", label="Training")
ax.plot(train_sizes, valid_rmse, marker="o", label="Validierung")

ax.set_title("Lernkurve: RMSE bei wachsender Trainingsmenge")
ax.set_xlabel("Anzahl Trainingsbeispiele")
ax.set_ylabel("RMSE, kleiner ist besser")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()



### **3.3. Modelle fair vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_03_03.jpg?v=1787643902" width="250">



>* Gleiche Trainings- und Testdaten für alle Modelle
>* Pipelines verhindern Datenleckage und unrealistische Ergebnisse

>* Mehrere Kennzahlen gemeinsam bewerten
>* Fehlermuster und praktische Folgen beachten

>* Genauigkeit gegen Aufwand und Laufzeit abwägen
>* Lernkurven zeigen Überanpassung und Datenbedarf



In [ ]:
#@title Python-Code - Modelle fair vergleichen

# Wir vergleichen Regressionsmodelle unter gleichen Bedingungen.
# Pipelines verhindern Datenleckage beim Skalieren.
# Kennzahlen und Laufzeit zeigen faire Unterschiede.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Regressionsdatensatz.
data = load_diabetes()
X = data.data
y = data.target

# Diese Prüfung macht die Datengröße bewusst sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Beide Modelle erhalten exakt dieselbe Aufteilung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Ein Dummy-Modell dient als einfache Vergleichsbasis.
baseline_model = DummyRegressor(strategy="mean")
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

# Wir messen Trainingszeit mit einer Colab-freundlichen Uhr.
start_time = pd.Timestamp.now()
baseline_model.fit(X_train, y_train)
baseline_seconds = (pd.Timestamp.now() - start_time).total_seconds()

# Die Pipeline skaliert nur anhand der Trainingsdaten.
start_time = pd.Timestamp.now()
ridge_model.fit(X_train, y_train)
ridge_seconds = (pd.Timestamp.now() - start_time).total_seconds()

# Vorhersagen entstehen auf denselben zurückgehaltenen Testdaten.
baseline_pred = baseline_model.predict(X_test)
ridge_pred = ridge_model.predict(X_test)

# Mehrere Kennzahlen zeigen verschiedene Qualitätsaspekte.
results = pd.DataFrame(
    {
        "Modell": ["Mittelwert-Basis", "Ridge-Pipeline"],
        "MAE": [
            mean_absolute_error(y_test, baseline_pred),
            mean_absolute_error(y_test, ridge_pred),
        ],
        "RMSE": [
            mean_squared_error(y_test, baseline_pred) ** 0.5,
            mean_squared_error(y_test, ridge_pred) ** 0.5,
        ],
        "R2": [r2_score(y_test, baseline_pred), r2_score(y_test, ridge_pred)],
        "Sekunden": [baseline_seconds, ridge_seconds],
    }
)

# Gerundete Werte sind für Anfänger leichter lesbar.
rounded_results = results.round({"MAE": 1, "RMSE": 1, "R2": 3, "Sekunden": 4})
print("scikit-learn Version:", sklearn.__version__)
print(rounded_results.to_string(index=False))

# Das Diagramm vergleicht den verständlichen absoluten Fehler.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(rounded_results["Modell"], rounded_results["MAE"], color=["gray", "steelblue"])
ax.set_title("Fairer Testvergleich: niedrigerer MAE ist besser")
ax.set_xlabel("Modell")
ax.set_ylabel("MAE in Zielwert-Einheiten")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Regression vergleichen**</font>


In this lecture, you learned to:
- Trainieren lineare, regularisierte und robuste Regressionsmodelle in Pipelines. 
- Vergleichen nichtlineare Regressoren wie k-NN, Bäume, Random Forest, Boosting und SVR. 
- Bewerten Regressionsmodelle mit MAE, RMSE, R², Lernkurven und Laufzeit. 

In the next Lecture (Lecture B), we will go over 'Klassifikation vergleichen'